In [2]:
# ============================================================
# CharBot domain generator (Colab-ready)
# ------------------------------------------------------------
# Implements Algorithm 1 from the CharBot paper:
#  - pick a benign domain
#  - pick two SLD positions
#  - substitute two DNS-valid characters that differ from originals
#  - attach a TLD chosen from a fixed list
# Notes:
#  - comments in English
#  - simple and reproducible
# ============================================================

import re
import random
import string
from typing import List, Tuple, Iterable, Set, Optional
from dataclasses import dataclass

# -----------------------------
# Config
# -----------------------------
DNS_VALID_CHARS = string.ascii_lowercase + string.digits + "-"  # allowed for SLD substitutions
TLD_POOL = [
    "com","at","uk","pl","be","biz","co","jp","cz","de","eu","fr","info","it",
    "ru","lv","me","name","net","nz","org","us"
]  # as listed in the paper

SLD_MIN_LEN = 6  # paper uses SLD length >= 6 for the benign list

# -----------------------------
# Helpers
# -----------------------------
SLD_RE = re.compile(r"^[a-z0-9-]+$")  # basic SLD sanity
DOMAIN_RE = re.compile(r"^(?=.{1,253}$)([a-z0-9-]{1,63}\.)+[a-z]{2,63}$")

def split_domain(domain: str) -> Tuple[str, str]:
    """
    Split a domain into (SLD, TLD).
    Assumes input like example.com or foo.bar (no subdomains for training list).
    If there are subdomains, keep the last two labels.
    """
    labels = domain.strip().lower().split(".")
    labels = [lbl for lbl in labels if lbl]  # remove empty pieces
    if len(labels) < 2:
        raise ValueError(f"Not a valid domain with TLD: {domain}")
    sld, tld = labels[-2], labels[-1]
    return sld, tld

def clean_benign_domains(raw_domains: Iterable[str], sld_min_len: int = SLD_MIN_LEN) -> List[str]:
    """
    Keep one domain per line, pick last two labels, filter SLD length >= sld_min_len,
    and keep only ASCII lowercase letters, digits, and dash in SLD.
    """
    cleaned = []
    seen: Set[str] = set()
    for d in raw_domains:
        d = d.strip().lower()
        if not d:
            continue
        try:
            sld, tld = split_domain(d)
        except ValueError:
            continue
        if len(sld) < sld_min_len:
            continue
        if not SLD_RE.match(sld):
            # if SLD has other chars, skip
            continue
        # recontruct canonical base "sld.tld"
        dom = f"{sld}.{tld}"
        if dom not in seen:
            seen.add(dom)
            cleaned.append(dom)
    return cleaned

@dataclass
class CharBotConfig:
    tld_pool: List[str] = None
    charset: str = DNS_VALID_CHARS

    def __post_init__(self):
        if self.tld_pool is None:
            self.tld_pool = list(TLD_POOL)

def substitute_two_chars(s: str, rng: random.Random, charset: str) -> str:
    """
    Substitute two (possibly distinct) positions in SLD with new chars from charset.
    New chars must differ from originals.
    """
    if len(s) < 2:
        # if SLD length is 1, duplicate substitution on same index
        idxs = [0, 0]
    else:
        i = rng.randrange(len(s))
        j = rng.randrange(len(s))
        # ensure two picks (can be equal by design, but picking distinct is closer to paper intent)
        # If equal, keep; the second replacement will still enforce char != original char
        idxs = [i, j]

    s_list = list(s)
    for idx in idxs:
        original = s_list[idx]
        # pick a replacement different from original
        choices = [c for c in charset if c != original]
        s_list[idx] = rng.choice(choices)
    return "".join(s_list)

def charbot_once(benign_base: str, cfg: CharBotConfig, rng: random.Random) -> str:
    """
    Generate one CharBot domain from a benign base.
    benign_base is 'sld.tld' from the benign list. We only modify the SLD.
    """
    sld, _ = split_domain(benign_base)
    # step 2 and 4 from the paper
    new_sld = substitute_two_chars(sld, rng, cfg.charset)
    new_tld = rng.choice(cfg.tld_pool)
    return f"{new_sld}.{new_tld}"

def charbot_batch(
    benign_domains: List[str],
    n_samples: int,
    seed: Optional[int] = None,
    cfg: Optional[CharBotConfig] = None
) -> List[str]:
    """
    Generate n_samples CharBot domains.
    - benign_domains: cleaned benign list like Alexa or Tranco
    - seed: for reproducibility
    """
    if cfg is None:
        cfg = CharBotConfig()
    rng = random.Random(seed)

    out = []
    for _ in range(n_samples):
        base = rng.choice(benign_domains)  # step 1
        dga = charbot_once(base, cfg, rng)
        out.append(dga)
    return out

# -----------------------------
# Optional: check registration status via DNS (slow)
# -----------------------------
# You can use this to estimate the fraction of NXDomain responses on a small sample.
# It increases runtime because it hits the resolver.
def check_dns_nx(domains: List[str], max_check: int = 100) -> List[Tuple[str, bool]]:
    """
    Resolve up to max_check domains and return (domain, is_nx) pairs.
    True means NXDOMAIN or similar failure, False means it resolved.
    """
    try:
        import dns.resolver
        import dns.exception
    except Exception as e:
        raise RuntimeError("Install dnspython first: !pip install dnspython") from e

    results = []
    resolver = dns.resolver.Resolver()
    resolver.lifetime = 2.0
    resolver.timeout = 2.0

    for d in domains[:max_check]:
        try:
            _ = resolver.resolve(d, "A")
            results.append((d, False))  # resolved
        except Exception:
            results.append((d, True))   # NX or other failure
    return results


In [3]:
# ============================================================
# Load benign domains (upload a file or paste a small list)
# ============================================================

# Option 1: upload a text file in the Colab left panel (Files) and set the path here.
BENIGN_PATH = "/content/benign_domains.txt"  # one domain per line

# Example fallback list if you do not have a file yet:
FALLBACK_BENIGN = [
    "google.com",
    "youtube.com",
    "facebook.com",
    "wikipedia.org",
    "amazon.com",
    "twitter.com",
    "instagram.com",
    "linkedin.com",
    "netflix.com",
    "reddit.com",
]

def load_benign(path: str) -> List[str]:
    try:
        with open(path, "r", encoding="utf-8") as f:
            raw = f.readlines()
        print(f"Loaded {len(raw)} lines from {path}")
        return clean_benign_domains(raw, sld_min_len=SLD_MIN_LEN)
    except FileNotFoundError:
        print("File not found. Using fallback list for a quick demo.")
        return clean_benign_domains(FALLBACK_BENIGN, sld_min_len=SLD_MIN_LEN)

benign_list = load_benign(BENIGN_PATH)
print(f"Benign list after cleaning: {len(benign_list)} domains (min SLD len {SLD_MIN_LEN})")
benign_list[:10]


File not found. Using fallback list for a quick demo.
Benign list after cleaning: 10 domains (min SLD len 6)


['google.com',
 'youtube.com',
 'facebook.com',
 'wikipedia.org',
 'amazon.com',
 'twitter.com',
 'instagram.com',
 'linkedin.com',
 'netflix.com',
 'reddit.com']

In [4]:
# ============================================================
# Generate CharBot domains
# ============================================================

N_SAMPLES = 1000       # how many DGA domains you want
SEED = 20250201        # any integer; choose date-based seed for reproducibility

cfg = CharBotConfig()
charbot_domains = charbot_batch(benign_list, n_samples=N_SAMPLES, seed=SEED, cfg=cfg)

# Show a preview
for d in charbot_domains[:20]:
    print(d)

# Save to disk
OUT_PATH = "/content/charbot_domains.txt"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(charbot_domains))
print(f"\nSaved {len(charbot_domains)} domains to {OUT_PATH}")


faneblok.ru
tw3tte3.eu
netflix.cz
inytagrac.jp
wixipe1ia.pl
re7diq.us
twitttr.fr
7nstagbam.co
wetfli1.net
instagram.pl
linked67.me
recdqt.net
twit7e4.fr
redhit.be
qeoflix.us
99azon.us
goog7e.net
wiktpe4ia.at
0ougle.eu
netfliu.jp

Saved 1000 domains to /content/charbot_domains.txt
